# Table 1 — kardemumma tools and libraries

Input, operations, and output for each command-line tool and major library module.
This table mirrors the structure of the [API reference HTML](https://github.com/thanadol-git/kardemumma/releases/latest/download/kardemumma_api.html) shipped with each release.

Run all cells to render the table and export `paper/table1_tools.html` (and optional CSV/LaTeX).

In [ ]:
from pathlib import Path

import pandas as pd

OUT_DIR = Path(".").resolve()

TOOLS = [
    # --- CLI pipeline ---
    {
        "Category": "CLI",
        "Tool": "kardemumma-preview",
        "Input": "Skyline CSV; SDRF `.tsv`; optional `--dotp` threshold",
        "Operations": "Import and validate files; detect QC samples; filter by library dot product; summarise peptide/protein counts; plot dot-product distribution and heavy/light scatter",
        "Output": "`report_summary.txt`, `skyline_raw.csv`, `sdrf.csv`, `peptide_counts_raw.csv`, `qc_samples.txt`, `dot_product_distribution.pdf`, `heavy_light_scatter.pdf`",
    },
    {
        "Category": "CLI",
        "Tool": "kardemumma-cutoff",
        "Input": "Skyline CSV; SDRF `.tsv`; cutoffs (`--dotp`, `--light-cutoff`, `--heavy-cutoff`, `--pool-value`, etc.)",
        "Operations": "QC removal; dot-product filter; peptide count filtering; merge Skyline with SDRF; pool CV analysis; select normalization peptides; compute plate conversion factors; batch-correct ratios by plate",
        "Output": "`skyline_merge_adj.csv`, `conversion_factors.csv`, `interplate_cv.csv`, pool/CV PDFs, `cutoff_params.json`",
    },
    {
        "Category": "CLI",
        "Tool": "kardemumma-report",
        "Input": "Output directory from cutoff (contains `skyline_merge_adj.csv`); SDRF `.tsv` (qRePS lot in `comment[ProteomEdge]`)",
        "Operations": "Fetch qRePS table and FASTA from ProteomEdge; compute absolute protein concentrations; generate concentration, median, PCA, and UMAP plots",
        "Output": "`qreps_table.csv`, `fasta.csv`, `abs_df.csv`, `protein_concentrations.pdf`, `protein_median_concentrations.pdf`, `pca.pdf`, `umap.pdf`",
    },
    {
        "Category": "CLI",
        "Tool": "kardemumma-ipynb",
        "Input": "Skyline CSV; SDRF `.tsv`; `--group-a` / `--group-b` labels; optional analysis parameters",
        "Operations": "Scaffold a ratio-analysis Jupyter notebook (import → QC → merge → pool normalization → absolute quant → differential expression → enrichment)",
        "Output": "Pre-filled `.ipynb` notebook (default `ratio_analysis.ipynb`)",
    },
    {
        "Category": "CLI",
        "Tool": "kardemumma-docs",
        "Input": "Installed `kardemumma` source tree",
        "Operations": "Parse public API from `src/kardemumma`; render HTML function reference; optional PDF via headless Chrome",
        "Output": "`kardemumma_api.html`, `kardemumma_api.pdf`",
    },
    # --- Import & merge ---
    {
        "Category": "Import",
        "Tool": "ImportSkylineFile",
        "Input": "Skyline report CSV (required columns: Precursor, Replicate, File Name, Peptide Sequence, RatioLightToHeavy, Library Dot Product, …)",
        "Operations": "Validate column names and dtypes; load long-format transition table",
        "Output": "pandas DataFrame (`skyline_data`)",
    },
    {
        "Category": "Import",
        "Tool": "ImportSDRFFile",
        "Input": "Targeted SDRF `.tsv` (`source name`, `comment[data file]`, `characteristics[...]`, `factor value[...]`)",
        "Operations": "Load sample metadata; extract qRePS lot from `comment[ProteomEdge]`; suggest QC sample names",
        "Output": "pandas DataFrame (`sdrf_data`); lot number string",
    },
    {
        "Category": "Import",
        "Tool": "MergeFiles.merge_files",
        "Input": "Skyline DataFrame; SDRF DataFrame; optional peptide allow-list",
        "Operations": "Left-join Skyline `File Name` to SDRF `comment[data file]`; attach plate, sample, and factor columns",
        "Output": "Merged long-format DataFrame (`skyline_merge`)",
    },
    # --- SDRF utilities ---
    {
        "Category": "SDRF",
        "Tool": "validate_sdrf",
        "Input": "SDRF `.tsv` file path",
        "Operations": "Run `parse_sdrf validate-sdrf` (sdrf-pipelines)",
        "Output": "`(success: bool, message: str)`",
    },
    {
        "Category": "SDRF",
        "Tool": "readout_ms_type",
        "Input": "SDRF `.tsv` file path",
        "Operations": "Read `comment[proteomics data acquisition method]` values",
        "Output": "List of unique MS acquisition types",
    },
    # --- QC & filtering ---
    {
        "Category": "QC / filtering",
        "Tool": "filter_library_dot_product",
        "Input": "Skyline long DataFrame; dot-product threshold",
        "Operations": "Keep rows above library dot product cutoff; drop missing areas; pivot to heavy/light wide format",
        "Output": "Filtered pivoted DataFrame (`skyline_clean`)",
    },
    {
        "Category": "QC / filtering",
        "Tool": "summarise_peptide_counts / filter_peptide_counts",
        "Input": "Pivoted peptide table; light/heavy count cutoffs",
        "Operations": "Count detected light/heavy replicates per peptide; apply minimum count filters",
        "Output": "Peptide count summary; filtered peptide allow-list",
    },
    # --- Normalization ---
    {
        "Category": "Normalization",
        "Tool": "calculate_intra_plate_cv / calculate_inter_plate_cv",
        "Input": "Pool-sample merged DataFrame; plate column",
        "Operations": "Compute intra- and inter-plate %CV per peptide across pool replicates",
        "Output": "Peptide-level CV statistics; inter-plate CV ranking",
    },
    {
        "Category": "Normalization",
        "Tool": "get_plate_conversion_factors / adjust_ratio_by_plate",
        "Input": "Pool peptides; plate labels; merged ratio table",
        "Operations": "Select low-CV normalization peptides; estimate per-plate conversion factors; adjust `RatioLightToHeavy`",
        "Output": "Conversion factor dict; batch-corrected DataFrame (`skyline_merge_adj`)",
    },
    # --- Absolute quantification ---
    {
        "Category": "Absolute quant",
        "Tool": "fetch_qreps_table / fetch_fasta",
        "Input": "ProteomEdge lot number or lot URL (from SDRF `comment[ProteomEdge]`)",
        "Operations": "Download qRePS spike-in table and target FASTA from ProteomEdge",
        "Output": "qRePS DataFrame (`qRePS`, `Amount per well [pmol]`, …); FASTA DataFrame (`id`, `sequence`)",
    },
    {
        "Category": "Absolute quant",
        "Tool": "get_absolute_conc",
        "Input": "qRePS table; batch-corrected Skyline merge (`RatioLightToHeavy`, `Protein Name`, `File Name`)",
        "Operations": "Match qRePS IDs; multiply ratio × spike amount; pivot to wide concentration matrix",
        "Output": "Wide DataFrame indexed by qRePS / peptide / protein; columns = sample files; values = `Protein conc [pmol]`",
    },
    # --- Downstream ---
    {
        "Category": "Downstream",
        "Tool": "DownStream",
        "Input": "Wide `abs_df`; SDRF table; two group labels (`group_a`, `group_b`)",
        "Operations": "Melt concentrations; merge SDRF metadata; Welch's t-test per peptide/protein; Benjamini–Hochberg adjustment; volcano plot",
        "Output": "Differential expression results (`log2fc`, `p_value`, `padj`); optional volcano figure",
    },
    {
        "Category": "Downstream",
        "Tool": "enrichment_analysis",
        "Input": "Hit ID list; pathway gene-set dictionary; background ID universe",
        "Operations": "Hypergeometric (Fisher's exact) enrichment test",
        "Output": "Enrichment table (`pathway`, `overlap`, `p_value`, `padj`)",
    },
    # --- Batch correction (advanced) ---
    {
        "Category": "Batch correction",
        "Tool": "permanova_batch_effects / correct_ratio_by_factors / correct_ratio_by_irt",
        "Input": "Merged ratio table; batch/plate factor columns; optional iRT offsets",
        "Operations": "PERMANOVA batch testing; median-based factor correction; iRT retention-time correction",
        "Output": "Batch-effect statistics; corrected ratio DataFrame",
    },
    # --- OpenSWATH (optional) ---
    {
        "Category": "OpenSWATH",
        "Tool": "import_openswath_file / get_ratio",
        "Input": "OpenSWATH TSV export",
        "Operations": "Import transition report; filter best peak groups; compute light/heavy ratios",
        "Output": "Ratio table compatible with downstream QC modules",
    },
    # --- Visualization (representative) ---
    {
        "Category": "Visualization",
        "Tool": "plot_* / map_peptide_sequence",
        "Input": "QC tables, pool data, or wide `abs_df`; SDRF metadata for grouping",
        "Operations": "Library dot-product distributions; pool boxplots; CV KDE; peptide concentration by group; protein sequence maps; PCA/UMAP",
        "Output": "matplotlib/seaborn figures (displayed or saved as PDF)",
    },
]

table1 = pd.DataFrame(TOOLS, columns=["Category", "Tool", "Input", "Operations", "Output"])
table1

In [ ]:
def render_tools_table_html(df: pd.DataFrame, title: str = "kardemumma — tools and libraries") -> str:
    """Render an API-style HTML table (similar to kardemumma_api.html)."""
    css = """
    <style>
      body { font-family: system-ui, sans-serif; margin: 2rem; color: #1a1a1a; }
      h1 { font-size: 1.4rem; margin-bottom: 0.25rem; }
      p.sub { color: #555; margin-top: 0; }
      table { border-collapse: collapse; width: 100%; font-size: 0.85rem; }
      th { background: #2c5282; color: #fff; text-align: left; padding: 0.55rem 0.65rem; }
      td { border: 1px solid #ddd; padding: 0.5rem 0.65rem; vertical-align: top; }
      tr:nth-child(even) { background: #f7fafc; }
      td.tool { font-family: ui-monospace, monospace; font-weight: 600; white-space: nowrap; }
      td.cat { color: #4a5568; font-size: 0.8rem; }
    </style>
    """
    rows = []
    for _, row in df.iterrows():
        rows.append(
            f"<tr>"
            f"<td class='cat'>{row['Category']}</td>"
            f"<td class='tool'>{row['Tool']}</td>"
            f"<td>{row['Input']}</td>"
            f"<td>{row['Operations']}</td>"
            f"<td>{row['Output']}</td>"
            f"</tr>"
        )
    body = "\n".join(rows)
    return f"""<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><title>{title}</title>{css}</head>
<body>
<h1>{title}</h1>
<p class="sub">Input, operations, and output for kardemumma CLI commands and library modules.</p>
<table>
<thead><tr><th>Category</th><th>Tool</th><th>Input</th><th>Operations</th><th>Output</th></tr></thead>
<tbody>
{body}
</tbody></table></body></html>"""


html = render_tools_table_html(table1)
html_path = OUT_DIR / "table1_tools.html"
html_path.write_text(html, encoding="utf-8")
print(f"Wrote {html_path}")

from IPython.display import HTML, display
display(HTML(html))

In [ ]:
# Export for manuscript / supplementary materials
csv_path = OUT_DIR / "table1_tools.csv"
tex_path = OUT_DIR / "table1_tools.tex"

table1.to_csv(csv_path, index=False)
print(f"Wrote {csv_path}")

try:
    latex = table1.to_latex(
        index=False,
        column_format="llp{3.5cm}p{4.5cm}p{3.5cm}",
        caption="kardemumma tools and libraries: input, operations, and output.",
        label="tab:kardemumma-tools",
        escape=True,
    )
    tex_path.write_text(latex, encoding="utf-8")
    print(f"Wrote {tex_path}")
except ImportError:
    print("Install jinja2 for LaTeX export: pip install jinja2")

## Optional: regenerate full API reference

The release API HTML lists every public function with signatures and docstrings. From the repo root:

```bash
kardemumma-docs --out-dir paper/
# → paper/kardemumma_api.html, paper/kardemumma_api.pdf
```

Use **Table 1** (this notebook) for the paper methods summary; link or attach **kardemumma_api.html** as supplementary detail.